In [ ]:
import numpy as np
import matplotlib as plt
import pandas as pd

from google.colab import data_table
data_table.enable_dataframe_formatter()

# Create dataframe of morphological data

In [ ]:
#snake ID numbers
snake_IDs = ["2018DP01","2018DP02","2018DP03","2018DC04","2018DC05",
             "2018DP06","2019DC07","2019DP08","2019DP09","2019DP10",
             "2019DP11","2019DP12","2019DP13","2019DP15",
             "2019DP16","2019DP17","2019DP18","2019DP19","2019DP20"]

#measured mass, in grams, of each snake's mass
snake_mass = [185.1,
              267.2,
              84.8,
              47.7,
              85.5,
              261.3,
              14.5,
              301.1,
              161.7,
              37.1,
              27.8,
              12.5,
              23.9,
              72.3,
              83.4,
              76.5,
              5.1,
              5.3,
              133.4]

#mass of each snake in KGs, excluding tail (scaled based on dissected specimen)
snake_mass_scaled = [i/1000*1077.5/1157.7 for i in snake_mass] #total mass of snake segments was 1157.7, mass of just SVL sections was 1077.5

#snout-vent length of each snake in meters
snake_SVL = [1.13988, 1.274433333, 0.8897333333, 0.77373, 0.842583333,
             1.243746667, 0.4619933333, 1.19094, 1.09, 0.65, 0.59, 0.43, 0.55,
             0.8699200, 0.7698133333, 0.81231176, 0.29534, 0.3064466667, 0.8949533333]



DF = pd.DataFrame(data=list(zip(snake_IDs, snake_mass, snake_mass_scaled, snake_SVL)), columns=["ID", "Mass", "SVL_Mass", "SVL"])
DF

,ID,Mass,SVL_Mass,SVL
0,2018DP01,185.1,0.172277,1.139880
1,2018DP02,267.2,0.248690,1.274433
2,2018DP03,84.8,0.078925,0.889733
3,2018DC04,47.7,0.044396,0.773730
4,2018DC05,85.5,0.079577,0.842583
5,2018DP06,261.3,0.243198,1.243747
6,2019DC07,14.5,0.013496,0.461993
7,2019DP08,301.1,0.280241,1.190940
8,2019DP09,161.7,0.150498,1.090000
9,2019DP10,37.1,0.034530,0.650000


# Create scaled data based on dissected snake

Based on a single dissection, and scaling relative to SVL and mass of that specimen, this section creates entries for each snake of cross sectional area (CSAmax), vertebral length (L), lever arm (a), and mass of a vertebral unit (m)

We also set constants for p0, p, t, p_point, t_point, and g.

*   p0: peak isometric muscle stress
*   p: A list of potential multi-articular span values based off the literature
*   t: A list of potential tendon ratio values based of the literate
*   p_point: our best guess point estimate as to the p value in d. punctulatus
*   t_point: our best guess point estimate as to the t value in d. punctulatus


For details of why these parameters, see Astley, Henry C. "The biomechanics of multi-articular muscle–tendon systems in snakes." Integrative and Comparative Biology 60.1 (2020): 140-155.

In [ ]:
#reference data from dissected snake
d_svl = 0.983
d_L = 0.003159553813                  #vertebral length, m
d_a = 0.003394850295                  #lever arm, m
d_CSAmax = 1.156561818/(1000*1000)    #CSA in mm squared - converted to meters squared
d_mass =  1077.5/1157.7 * 138.6/1000  #mass in kg of SVL section
d_m_vert = d_mass/d_svl*d_L           #total mass/length * vertebral length


#scale reference data from dissected snake to each individual in our trials
DF['L_vertebral_length'] = d_L/d_svl*DF['SVL']
DF['a_lever_arm'] = d_a/d_svl*DF['SVL']
DF['CSAmax'] = d_CSAmax/d_svl**2*DF['SVL']**2
DF['m_verterbal_unit'] = DF['L_vertebral_length']*DF['SVL_Mass']/DF['SVL'] #length of vertebral unit * SVL_mass/SVL

#set constants
p0 = 300000                     #Peak isometric muuscle stress, N/m^2
g = 9.8                         #m/s^2; acceleration due to gravity

#given our uncertainty about the precise values for D. punctulatus, use a range representing values found in the sub-family.
p = list(range(9, 42))          #multi-articular span: (the number of vertebral units spanned by the SSP muscle-tendon complex, overall)
t = np.arange(0.43, 0.94, 0.04) #tendon ratio: the number of vertebral units spanned by the tendon from the SSP muscle-tendon complex.

#also doing a point estimate, based on values from D. pictus (Nicodemo, 2012)
p_point = 34
t_point = 0.91


print(t)


[0.43 0.47 0.51 0.55 0.59 0.63 0.67 0.71 0.75 0.79 0.83 0.87 0.91]


# Create function; estimate point estimate for buckling failure.
Utilizing the equation in Astley, 2020 and the scaled values, as well as p and t point estimates, to create a function and apply it to generate expected buckling distances.



In [ ]:
#create equation
def buckle_estimate(p, t, L, a, CSA, M):
  radical1 = 0.5*(p-1)**2
  radical2_top = 8*a*p0*CSA
  radical2_bottom = M*g*L*(1-t)

  Max_i = 0.5*(-p+1+np.sqrt(radical1+radical2_top/radical2_bottom))
  buckle = L*Max_i

  return buckle

DF['buckle_estimate_cm'] = buckle_estimate(p_point, t_point, DF['L_vertebral_length'], DF['a_lever_arm'], DF['CSAmax'], DF['m_verterbal_unit'])*100
DF[['ID','SVL','buckle_estimate_cm']]

,ID,SVL,buckle_estimate_cm
0,2018DP01,1.139880,11.096361
1,2018DP02,1.274433,11.160960
2,2018DP03,0.889733,10.591645
3,2018DC04,0.773730,11.240019
4,2018DC05,0.842583,9.245976
5,2018DP06,1.243747,10.679156
6,2019DC07,0.461993,7.445471
7,2019DP08,1.190940,8.576931
8,2019DP09,1.090000,10.966051
9,2019DP10,0.650000,8.853590


# Calculate range of potential buckling values
Utilizing the equation, scaled values, and our list of potential t and p values, calculate a range of potential buckling distances for d. punctulatus

In [ ]:
#create max and min
maxes = []
mins = []

for index, row in DF.iterrows():
  L = row['L_vertebral_length']
  a = row['a_lever_arm']
  CSA = row['CSAmax']
  M = row['m_verterbal_unit']

  all = []

  for i in p:
    for j in t:
      be = buckle_estimate(i, j, L, a, CSA, M)*100
      all.append(be)

  maxes.append(max(all))
  mins.append(min(all))

DF['max_buckle_cm'] = maxes
DF['min_buckle_cm'] = mins
DF[['ID','SVL','buckle_estimate_cm','max_buckle_cm','min_buckle_cm']]

,ID,SVL,buckle_estimate_cm,max_buckle_cm,min_buckle_cm
0,2018DP01,1.139880,11.096361,15.166877,1.060298
1,2018DP02,1.274433,11.160960,15.671057,0.788293
2,2018DP03,0.889733,10.591645,13.820250,1.464463
3,2018DC04,0.773730,11.240019,14.088229,1.969645
4,2018DC05,0.842583,9.245976,12.284266,1.125014
5,2018DP06,1.243747,10.679156,15.073046,0.702339
6,2019DC07,0.461993,7.445471,9.157847,1.434409
7,2019DP08,1.190940,8.576931,12.717591,0.165788
8,2019DP09,1.090000,10.966051,14.869081,1.128950
9,2019DP10,0.650000,8.853590,11.235661,1.450279


# Final Table

In [ ]:
DF

,ID,Mass,SVL_Mass,SVL,L_vertebral_length,a_lever_arm,CSAmax,m_verterbal_unit,buckle_estimate_cm,max_buckle_cm,min_buckle_cm
0,2018DP01,185.1,0.172277,1.139880,0.003664,0.003937,1.555178e-06,0.000554,11.096361,15.166877,1.060298
1,2018DP02,267.2,0.248690,1.274433,0.004096,0.004401,1.943999e-06,0.000799,11.160960,15.671057,0.788293
2,2018DP03,84.8,0.078925,0.889733,0.002860,0.003073,9.475051e-07,0.000254,10.591645,13.820250,1.464463
3,2018DC04,47.7,0.044396,0.773730,0.002487,0.002672,7.165404e-07,0.000143,11.240019,14.088229,1.969645
4,2018DC05,85.5,0.079577,0.842583,0.002708,0.002910,8.497429e-07,0.000256,9.245976,12.284266,1.125014
5,2018DP06,261.3,0.243198,1.243747,0.003998,0.004295,1.851508e-06,0.000782,10.679156,15.073046,0.702339
6,2019DC07,14.5,0.013496,0.461993,0.001485,0.001596,2.554661e-07,0.000043,7.445471,9.157847,1.434409
7,2019DP08,301.1,0.280241,1.190940,0.003828,0.004113,1.697624e-06,0.000901,8.576931,12.717591,0.165788
8,2019DP09,161.7,0.150498,1.090000,0.003503,0.003764,1.422050e-06,0.000484,10.966051,14.869081,1.128950
9,2019DP10,37.1,0.034530,0.650000,0.002089,0.002245,5.056948e-07,0.000111,8.853590,11.235661,1.450279
